[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/09_causal_attention.ipynb)

# 🔴 Hard: Causal Self-Attention

Implement **causal (masked) self-attention** — the attention used in GPT-style decoders.

Same as softmax attention, but each position can **only attend to itself and earlier positions** (no peeking at future tokens).

$$\text{scores}_{ij} = \begin{cases} \frac{Q_i \cdot K_j}{\sqrt{d_k}} & \text{if } j \le i \\ -\infty & \text{if } j > i \end{cases}$$

### Signature
```python
def causal_attention(Q, K, V):
    # Q, K, V: (batch, seq, d) → output: (batch, seq, d_v)
```

### Rules
- Do **NOT** use `F.scaled_dot_product_attention`
- Position $i$ can only attend to positions $\le i$
- You **may** use `torch.softmax`, `torch.bmm`, `torch.triu`

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [4]:
import torch
import math

e:\Graduate\MAC同步\文稿\1.code\TorchCode\.venv\Lib\site-packages\torch\_subclasses\functional_tensor.py:368: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


In [ ]:
# ✏️ YOUR IMPLEMENTATION HERE

def causal_attention(Q, K, V):
    d = Q.shape[-1]
    # shape (batch,seq_length,seq_length)
    score:torch.Tensor = Q @ K.transpose(1,2) / math.sqrt(d)
    mask = torch.ones(score.shape[-2:],dtype=torch.bool)
    # diagonal 从主对角线错开几位
    mask = torch.triu(mask,diagonal=1).bool()
    score.masked_fill_(mask,float('-inf'))
    score = torch.softmax(score,dim=-1)
    return score @ V
    

In [25]:
# 🧪 Debug
torch.manual_seed(0)
Q = torch.randn(1, 4, 8)
K = torch.randn(1, 4, 8)
V = torch.randn(1, 4, 8)
out = causal_attention(Q, K, V)
print("Output shape:", out.shape)          # (1, 4, 8)
print("Pos 0 == V[0]?", torch.allclose(out[:, 0], V[:, 0], atol=1e-5))  # should be True

Output shape: torch.Size([1, 4, 8])
Pos 0 == V[0]? True


In [26]:
from torch_judge import check
check('causal_attention')


🧪 Testing: Causal Self-Attention (Hard)
──────────────────────────────────────────────────
  ✅ [1/4] Output shape (0.5ms)
  ✅ [2/4] Future tokens don't affect past (4.8ms)
  ✅ [3/4] First position only sees itself (6.3ms)
  ✅ [4/4] Gradient flow (0.7ms)
──────────────────────────────────────────────────
  🎉 All 4 tests passed! (12.2ms total)
  Progress saved. Run status() to see your dashboard.

